In [35]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


import pandas as pd
import re
from difflib import SequenceMatcher

df = pd.read_csv("/kaggle/input/datasets/sudalairajkumar/indian-startup-funding/startup_funding.csv")
df.head()

/kaggle/input/datasets/sudalairajkumar/indian-startup-funding/startup_funding.csv


,Sr No,Date dd/mm/yyyy,Startup Name,Industry Vertical,SubVertical,City Location,Investors Name,InvestmentnType,Amount in USD,Remarks
0,1,09/01/2020,BYJU’S,E-Tech,E-learning,Bengaluru,Tiger Global Management,Private Equity Round,"20,00,00,000",NaN
1,2,13/01/2020,Shuttl,Transportation,App based shuttle service,Gurgaon,Susquehanna Growth Equity,Series C,"80,48,394",NaN
2,3,09/01/2020,Mamaearth,E-commerce,Retailer of baby and toddler products,Bengaluru,Sequoia Capital India,Series B,"1,83,58,860",NaN
3,4,02/01/2020,https://www.wealthbucket.in/,FinTech,Online Investment,New Delhi,Vinod Khatumal,Pre-series A,"30,00,000",NaN
4,5,02/01/2020,Fashor,Fashion and Apparel,Embroiled Clothes For Women,Mumbai,Sprout Venture Partners,Seed Round,"18,00,000",NaN


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3044 entries, 0 to 3043
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Sr No              3044 non-null   int64 
 1   Date dd/mm/yyyy    3044 non-null   object
 2   Startup Name       3044 non-null   object
 3   Industry Vertical  2873 non-null   object
 4   SubVertical        2108 non-null   object
 5   City  Location     2864 non-null   object
 6   Investors Name     3020 non-null   object
 7   InvestmentnType    3040 non-null   object
 8   Amount in USD      2084 non-null   object
 9   Remarks            419 non-null    object
dtypes: int64(1), object(9)
memory usage: 237.9+ KB


In [ ]:
#  1. Check missing values 
print(df.isnull().sum())
# Fill missing values
df["Industry Vertical"]  = df["Industry Vertical"].fillna("Other")
df["SubVertical"]        = df["SubVertical"].fillna("Other")
df["City  Location"]     = df["City  Location"].fillna("Unknown")
df["InvestmentnType"]    = df["InvestmentnType"].fillna("Undisclosed")
df.drop(columns=["Remarks"], inplace=True)  # 2625/3044 missing, not useful

# Verify
print(df.isnull().sum())

Sr No                   0
Date dd/mm/yyyy         0
Startup Name            0
Industry Vertical     171
SubVertical           936
City  Location        180
Investors Name         24
InvestmentnType         4
Amount in USD         960
Remarks              2625
dtype: int64
Sr No                  0
Date dd/mm/yyyy        0
Startup Name           0
Industry Vertical      0
SubVertical            0
City  Location         0
Investors Name        24
InvestmentnType        0
Amount in USD        960
dtype: int64


In [38]:
# Step 1: title case + strip
df["Investors Name"] = df["Investors Name"].str.strip().str.title()

# Step 2: remove parentheticals like (through crowdfunding)
df["Investors Name"] = df["Investors Name"].str.replace(r"\s*\(.*?\)", "", regex=True).str.strip()

# Before fuzzy dedup
print("Before:", df["Investors Name"].nunique(), "unique investors")

Before: 2376 unique investors


In [39]:
def fuzzy_deduplicate(series, threshold=0.88):
    names = series.dropna().unique().tolist()
    canonical, mapping = [], {}
    for name in sorted(names):
        if not canonical:
            canonical.append(name)
            mapping[name] = name
            continue
        best_match, best_score = None, 0
        for c in canonical:
            score = SequenceMatcher(None, name.lower(), c.lower()).ratio()
            if score > best_score:
                best_score, best_match = score, c
        if best_score >= threshold:
            mapping[name] = best_match
        else:
            canonical.append(name)
            mapping[name] = name
    return series.map(mapping)

df["Investors Name"] = fuzzy_deduplicate(df["Investors Name"])
print("After:", df["Investors Name"].nunique(), "unique investors")

After: 2292 unique investors


In [40]:
# City name duplicates
city_map = {
    "Bangalore": "Bengaluru",
    "Delhi":     "New Delhi",
    "Gurgaon":   "Gurugram",
    "Bombay":    "Mumbai",
}
df["City  Location"] = df["City  Location"].replace(city_map)
print(df["City  Location"].value_counts().head(10))


# # Final cleaned dataframe check
# print(df.shape)
# df.head()

City  Location
Bengaluru    841
Mumbai       567
New Delhi    455
Gurugram     337
Unknown      180
Pune         105
Hyderabad     99
Chennai       97
Noida         92
Ahmedabad     38
Name: count, dtype: int64


In [ ]:
# Amount: USD → INR Crore 
USD_TO_INR = 84.0  # 1 USD = 84 INR (approx current rate)

df["Amount in USD"] = df["Amount in USD"].str.replace(",", "", regex=False)
df["Amount in USD"] = pd.to_numeric(df["Amount in USD"], errors="coerce")


df["Amount in INR Crore"] = (df["Amount in USD"] * USD_TO_INR / 1e7).round(2)

print(df[["Amount in USD", "Amount in INR Crore"]].head(10))

   Amount in USD  Amount in INR Crore
0    200000000.0              1680.00
1      8048394.0                67.61
2     18358860.0               154.21
3      3000000.0                25.20
4      1800000.0                15.12
5      9000000.0                75.60
6    150000000.0              1260.00
7      6000000.0                50.40
8     70000000.0               588.00
9     50000000.0               420.00


In [ ]:
#  Date: string → pandas datetime 
df["Date dd/mm/yyyy"] = pd.to_datetime(
    df["Date dd/mm/yyyy"], 
    dayfirst=True, 
    errors="coerce"          # bad dates become NaT instead of crashing
)

# Rename for clarity
df.rename(columns={"Date dd/mm/yyyy": "Date"}, inplace=True)

# Extract useful columns for dashboard filters
df["Year"]  = df["Date"].dt.year
df["Month"] = df["Date"].dt.month_name()

print(df[["Date", "Year", "Month"]].head(10))
print("\nDate dtype:", df["Date"].dtype)

        Date    Year     Month
0 2020-01-09  2020.0   January
1 2020-01-13  2020.0   January
2 2020-01-09  2020.0   January
3 2020-01-02  2020.0   January
4 2020-01-02  2020.0   January
5 2020-01-13  2020.0   January
6 2020-01-10  2020.0   January
7 2019-12-12  2019.0  December
8 2019-12-06  2019.0  December
9 2019-12-03  2019.0  December

Date dtype: datetime64[ns]


In [ ]:
# df.to_csv("/kaggle/working/startup_funding_cleaned.csv", index=False)
# print("Saved ")

In [43]:
df.rename(columns={
    "Sr No"            : "sr_no",
    "Date dd/mm/yyyy"  : "date",
    "Startup Name"     : "startup",
    "Industry Vertical": "industry",
    "SubVertical"      : "sub_vertical",
    "City  Location"   : "city",
    "Investors Name"   : "investors",
    "InvestmentnType"  : "investment_type",
    "Amount in USD"    : "amount_usd",
    "Amount in INR Crore": "amount_inr_crore",
    "Year"             : "year",
    "Month"            : "month",
}, inplace=True)

print(df.columns.tolist())

['sr_no', 'Date', 'startup', 'industry', 'sub_vertical', 'city', 'investors', 'investment_type', 'amount_usd', 'amount_inr_crore', 'year', 'month']


In [44]:
df.head()

,sr_no,Date,startup,industry,sub_vertical,city,investors,investment_type,amount_usd,amount_inr_crore,year,month
0,1,2020-01-09,BYJU’S,E-Tech,E-learning,Bengaluru,Tiger Global Management,Private Equity Round,200000000.0,1680.00,2020.0,January
1,2,2020-01-13,Shuttl,Transportation,App based shuttle service,Gurugram,Susquehanna Growth Equity,Series C,8048394.0,67.61,2020.0,January
2,3,2020-01-09,Mamaearth,E-commerce,Retailer of baby and toddler products,Bengaluru,Sequoia Capital India,Series B,18358860.0,154.21,2020.0,January
3,4,2020-01-02,https://www.wealthbucket.in/,FinTech,Online Investment,New Delhi,Vinod Khatumal,Pre-series A,3000000.0,25.20,2020.0,January
4,5,2020-01-02,Fashor,Fashion and Apparel,Embroiled Clothes For Women,Mumbai,Sprout Venture Partners,Seed Round,1800000.0,15.12,2020.0,January


In [45]:
import re
df["startup"] = df["startup"].str.replace(
    r"https?://(?:www\.)?([^/]+).*", r"\1", regex=True
)

print(df["startup"].head())

0             BYJU’S
1             Shuttl
2          Mamaearth
3    wealthbucket.in
4             Fashor
Name: startup, dtype: object


In [ ]:
# df.to_csv("/kaggle/working/startup_funding_cleaned.csv", index=False)
# print("Saved")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3044 entries, 0 to 3043
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   sr_no             3044 non-null   int64         
 1   Date              3036 non-null   datetime64[ns]
 2   startup           3044 non-null   object        
 3   industry          3044 non-null   object        
 4   sub_vertical      3044 non-null   object        
 5   city              3044 non-null   object        
 6   investors         3020 non-null   object        
 7   investment_type   3044 non-null   object        
 8   amount_usd        2065 non-null   float64       
 9   amount_inr_crore  2065 non-null   float64       
 10  year              3036 non-null   float64       
 11  month             3036 non-null   object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(7)
memory usage: 285.5+ KB


In [47]:
df.describe()

,sr_no,Date,amount_usd,amount_inr_crore,year
count,3044.000000,3036,2.065000e+03,2065.000000,3036.000000
mean,1522.500000,2016-09-23 03:38:39.367588864,1.842990e+07,154.811133,2016.242754
min,1.000000,2015-01-02 00:00:00,1.600000e+04,0.130000,2015.000000
25%,761.750000,2015-11-04 00:00:00,4.700000e+05,3.950000,2015.000000
50%,1522.500000,2016-07-15 00:00:00,1.700000e+06,14.280000,2016.000000
75%,2283.250000,2017-06-12 06:00:00,8.000000e+06,67.200000,2017.000000
max,3044.000000,2020-01-13 00:00:00,3.900000e+09,32760.000000,2020.000000
std,878.871435,NaN,1.213734e+08,1019.536930,1.116610


In [49]:
df.sample(10)

,sr_no,Date,startup,industry,sub_vertical,city,investors,investment_type,amount_usd,amount_inr_crore,year,month
2779,2780,2015-05-13,MyCuteOffice,Online Office Rental,Other,Mumbai,Lead Angels,Seed Funding,NaN,NaN,2015.0,May
268,269,2018-05-22,Cashkumar,Consumer Internet,P2P Lending Platform,Bengaluru,Letsventure,Seed/ Angel Funding,735000.0,6.17,2018.0,May
2228,2229,2015-11-11,Delight Foods,Branded Food products online sales,Other,Bengaluru,"Lensbridge Capital, Mape Advisory Group, Fires...",Private Equity,600000.0,5.04,2015.0,November
788,789,2017-05-02,4tigo,Logistics,Truck Network company,Bengaluru,"Accel Partners, Nandan Nilekani,",Private Equity,10000000.0,84.00,2017.0,May
64,65,2019-07-03,Ola Cabs,Transport,Cabs,Kormangala,"Dig Investment Ab, Deshe Holdings, Samih Touka...",Series J,1000000.0,8.40,2019.0,July
3019,3020,2015-01-19,Yo Grad,Other,Other,Unknown,Hyderabad Angels,Seed Funding,16000.0,0.13,2015.0,January
51,52,2019-08-13,Uniphore,Customer Service Platform,Conversational AI,Palo Alto,March Capital Partners,Series C,51000000.0,428.40,2019.0,August
421,422,2018-01-29,The Wedding Brigade,Consumer internet,Online Services,Mumbai,Blume Ventures,Private Equity,1000000.0,8.40,2018.0,January
2690,2691,2015-06-05,kWatt Solutions,Renewable energy solutions,Other,Mumbai,NaN,Seed Funding,500000.0,4.20,2015.0,June
247,248,2018-06-22,WickedRide,Consumer Internet,Online Motorbike And Scooter Rental Platforms,Bengaluru,"Sequoia Capital, Accel Partners & Raghunandan ...",Private Equity,9100000.0,76.44,2018.0,June


In [50]:
# Fix column name
df.rename(columns={"Date": "date"}, inplace=True)

# Fix year dtype
df["year"] = df["year"].fillna(0).astype(int)

# Fix remaining NaN investors
df["investors"] = df["investors"].fillna("Undisclosed")

# Verify
print(df.isnull().sum())
print(df.dtypes)

sr_no                 0
date                  8
startup               0
industry              0
sub_vertical          0
city                  0
investors             0
investment_type       0
amount_usd          979
amount_inr_crore    979
year                  0
month                 8
dtype: int64
sr_no                        int64
date                datetime64[ns]
startup                     object
industry                    object
sub_vertical                object
city                        object
investors                   object
investment_type             object
amount_usd                 float64
amount_inr_crore           float64
year                         int64
month                       object
dtype: object


In [51]:
# 8 missing dates — fill with unknown marker
df["date"] = df["date"].fillna(pd.NaT)  # keep as NaT, fine for datetime

# 979 missing amounts — fill with 0
df["amount_usd"]        = df["amount_usd"].fillna(0)
df["amount_inr_crore"]  = df["amount_inr_crore"].fillna(0)

# 8 missing months/year
df["month"] = df["month"].fillna("Unknown")
df["year"]  = df["year"].replace(0, df["year"].mode()[0])

print(df.isnull().sum())

sr_no               0
date                8
startup             0
industry            0
sub_vertical        0
city                0
investors           0
investment_type     0
amount_usd          0
amount_inr_crore    0
year                0
month               0
dtype: int64


In [52]:
# df.to_csv("/kaggle/working/startup_funding_cleaned.csv", index=False)
# print("Saved ")

Saved 


In [55]:
df['investors']

0                 Tiger Global Management
1               Susquehanna Growth Equity
2                   Sequoia Capital India
3                          Vinod Khatumal
4                 Sprout Venture Partners
                      ...                
3039          Asia Pacific Internet Group
3040                       Karsemven Fund
3041       Exfinity Fund, Growx Ventures.
3042                           Makemytrip
3043    Uk Based Group Of Angel Investors
Name: investors, Length: 3044, dtype: object

In [54]:
investors_exploded = df["investors"].str.split(",").explode().str.strip()

print("Raw unique:", df["investors"].nunique())
print("True unique:", investors_exploded.nunique())
print("\nTop 10:")
print(investors_exploded.value_counts().head(10))

Raw unique: 2292
True unique: 3167

Top 10:
investors
3 Undisclosed Investors    106
Sequoia Capital             72
Accel Partners              69
Kalaari Capital             49
Blume Ventures              49
                            48
Saif Partners               47
Indian Angel Network        46
Undisclosed                 35
Nexus Venture Partners      31
Name: count, dtype: int64


In [56]:
df.head(
    
)

,sr_no,date,startup,industry,sub_vertical,city,investors,investment_type,amount_usd,amount_inr_crore,year,month
0,1,2020-01-09,BYJU’S,E-Tech,E-learning,Bengaluru,Tiger Global Management,Private Equity Round,200000000.0,1680.00,2020,January
1,2,2020-01-13,Shuttl,Transportation,App based shuttle service,Gurugram,Susquehanna Growth Equity,Series C,8048394.0,67.61,2020,January
2,3,2020-01-09,Mamaearth,E-commerce,Retailer of baby and toddler products,Bengaluru,Sequoia Capital India,Series B,18358860.0,154.21,2020,January
3,4,2020-01-02,wealthbucket.in,FinTech,Online Investment,New Delhi,Vinod Khatumal,Pre-series A,3000000.0,25.20,2020,January
4,5,2020-01-02,Fashor,Fashion and Apparel,Embroiled Clothes For Women,Mumbai,Sprout Venture Partners,Seed Round,1800000.0,15.12,2020,January


In [57]:
df['startup']

0                 BYJU’S
1                 Shuttl
2              Mamaearth
3        wealthbucket.in
4                 Fashor
              ...       
3039          Printvenue
3040            Graphene
3041      Mad Street Den
3042           Simplotel
3043    couponmachine.in
Name: startup, Length: 3044, dtype: object

In [64]:
print("Total unique startups:", df["startup"].nunique())

# See startups that appear more than once (multiple funding rounds)
print(df["startup"].value_counts().head(20))

Total unique startups: 2459
startup
Swiggy                  8
Ola Cabs                8
Paytm                   7
Medinfi                 6
NoBroker                6
Meesho                  6
UrbanClap               6
Nykaa                   6
Capital Float           5
Uniphore                5
Jugnoo                  5
Flipkart                5
Grofers                 5
Moglix                  5
Toppr                   5
Zomato                  4
BigBasket               4
Ola                     4
Byju\\xe2\\x80\\x99s    4
Udaan                   4
Name: count, dtype: int64


In [67]:
df[df["investors"].str.contains('Tiger Global Management')][["date", "startup", "industry", "amount_inr_crore", "investment_type"]].sort_values("date", ascending=False)

,date,startup,industry,amount_inr_crore,investment_type
0,2020-01-09,BYJU’S,E-Tech,1680.00,Private Equity Round
55,2019-08-22,INDwealth,FinTech,126.00,Venture Round
68,2019-07-11,Moglix,E-Commerce,504.00,Series D
85,2019-06-10,OkCredit,FinTech,130.20,Series A
93,2019-05-02,Zenoti,Saas,420.00,Series C
107,2019-04-11,CleverTap,SaaS,218.40,Series B
340,2018-03-22,Chargebee,Technology,151.20,Private Equity
547,2017-10-13,Chaayos,eCommerce,16.80,Private Equity
2036,2016-01-12,Shopclues,ECommerce,840.00,Private Equity
2140,2015-12-10,BlackBuck,Online Freight Services Aggregator,210.00,Private Equity


In [66]:
print(df.columns.tolist())

['sr_no', 'date', 'startup', 'industry', 'sub_vertical', 'city', 'investors', 'investment_type', 'amount_usd', 'amount_inr_crore', 'year', 'month']


In [70]:
df[df["investors"].str.contains('Tiger Global Management')].groupby('startup')['amount_inr_crore'].sum().sort_values(ascending=False)

startup
BYJU’S                   1680.00
Saavn                     840.00
Shopclues                 840.00
Delhivery                 714.00
Moglix                    504.00
Zenoti                    420.00
Zovi.com / Little App     420.00
Grey Orange               252.00
Zo Rooms                  252.00
CleverTap                 218.40
BlackBuck                 210.00
Chargebee                 151.20
OkCredit                  130.20
INDwealth                 126.00
Lybrate                    85.68
Grofers                    84.00
Razorpay                   75.60
Cube26                     64.68
Roposo.com                 42.00
LocalOye                   42.00
Vedantu                    42.00
Chaayos                    25.20
Name: amount_inr_crore, dtype: float64